# EDA — La Liga 2025-26 (Understat)

Chequeó rápido de los 4 CSVs producidos por `src/data/fetch.py` para informar el diseño de los agentes **Scout** y **Coach**.

Preguntas que respondemos:

1. ¿Los datos están íntegros (NaN, duplicados, schedule completo)?
2. ¿Cómo parseamos las posiciones de jugador?
3. ¿Qué rangos tienen las métricas tácticas (xG, PPDA, deep_completions)?
4. ¿Qué umbrales usar para describir "presión alta" o "ataque dominante" en los prompts?

In [1]:
from pathlib import Path
import pandas as pd

for candidate in [Path('data/processed'), Path('../data/processed')]:
    if candidate.exists():
        PROC = candidate.resolve()
        break

players = pd.read_csv(PROC / 'players.csv')
team_matches = pd.read_csv(PROC / 'team_matches.csv')
matches = pd.read_csv(PROC / 'matches.csv')
teams = pd.read_csv(PROC / 'teams.csv')

print(f'players: {len(players)} | team_matches: {len(team_matches)} | matches: {len(matches)} | teams: {len(teams)}')

players: 600 | team_matches: 380 | matches: 380 | teams: 20


## 1. Integridad

In [2]:
for name, df in [('players', players), ('team_matches', team_matches), ('matches', matches), ('teams', teams)]:
    nans = int(df.isna().sum().sum())
    dups = int(df.duplicated().sum())
    print(f'{name:14s}  NaN totales: {nans:5d}  |  duplicados: {dups}')

players         NaN totales:     0  |  duplicados: 0
team_matches    NaN totales:     0  |  duplicados: 0
matches         NaN totales:     0  |  duplicados: 0
teams           NaN totales:     0  |  duplicados: 0


In [3]:
matches['date'] = pd.to_datetime(matches['date'])
print('Rango fechas:', matches['date'].min().date(), '→', matches['date'].max().date())
print('Partidos con resultado:', int(matches['is_result'].sum()), '/', len(matches))
print('Partidos sin resultado (jornadas futuras):', int((~matches['is_result']).sum()))

Rango fechas: 2025-08-15 → 2026-05-24
Partidos con resultado: 380 / 380
Partidos sin resultado (jornadas futuras): 0


## 2. Posiciones

In [4]:
players['position'].value_counts().head(15)

position
D S        141
M S        114
F M S      100
S           71
F S         49
GK          37
D M S       35
D           24
D F M S     12
M            9
F            4
F M          2
GK S         2
Name: count, dtype: int64

In [5]:
def primary_position(pos):
    """Primera letra del código multi-posición de Understat."""
    if pd.isna(pos):
        return '?'
    return str(pos).strip().split()[0]

players['pos_main'] = players['position'].apply(primary_position)
players['pos_main'].value_counts()

pos_main
D     212
F     155
M     123
S      71
GK     39
Name: count, dtype: int64

**Lectura clave:** Understat agrega las posiciones a lo largo de la temporada. `"D F M S"` no es "posición compuesta"; significa que el jugador apareció como D, F, M y como S en distintos partidos.

**Implicación para el agente Coach:** las posiciones de Understat sirven para clasificación general (delantero vs defensor), pero **no para asignar XI en un partido concreto**. Para eso vamos a necesitar `soccerdata.ESPN.read_lineup()` cuando construyamos el Coach.

## 3. Minutos y xG por jugador

In [6]:
regulares = players[players['minutes'] >= 500]
print(f'Jugadores con ≥500 minutos: {len(regulares)} / {len(players)} ({100*len(regulares)/len(players):.0f}%)')
players['minutes'].describe().round(1)

Jugadores con ≥500 minutos: 415 / 600 (69%)


count     600.0
mean     1254.6
std       983.2
min         1.0
25%       299.5
50%      1170.0
75%      2046.5
max      3420.0
Name: minutes, dtype: float64

In [7]:
regulares.nlargest(15, 'xg')[['player', 'team', 'pos_main', 'minutes', 'goals', 'xg', 'xa']].reset_index(drop=True)

,player,team,pos_main,minutes,goals,xg,xa
0,Kylian Mbappe-Lottin,Real Madrid,F,2623,25,25.796529,7.240632
1,Vedat Muriqi,Mallorca,F,3150,23,21.630785,2.485310
2,Ante Budimir,Osasuna,F,2995,17,19.191765,1.712336
3,Ferrán Torres,Barcelona,F,2025,16,16.183372,3.157756
4,Robert Lewandowski,Barcelona,F,1630,14,16.066804,2.279529
5,Mikel Oyarzabal,Real Sociedad,F,2727,15,16.028234,5.957455
6,Georges Mikautadze,Villarreal,F,2155,13,15.062094,4.149702
7,Lamine Yamal,Barcelona,F,2291,16,14.664690,12.566995
8,Vinícius Júnior,Real Madrid,F,2865,16,14.403763,7.155190
9,Alexander Sørloth,Atletico Madrid,F,1955,13,13.927609,1.223736


## 4. Perfiles de equipo (PPDA, deep_completions)

In [8]:
teams[['team', 'partidos', 'puntos', 'ppda_promedio', 'deep_completions_total']].sort_values('ppda_promedio').reset_index(drop=True)

,team,partidos,puntos,ppda_promedio,deep_completions_total
0,Barcelona,38,94,7.313716,438
1,Elche,38,43,8.557275,206
2,Rayo Vallecano,38,50,8.865585,185
3,Sevilla,38,43,9.617936,201
4,Real Sociedad,38,46,9.848472,229
5,Athletic Club,38,45,10.510223,258
6,Alaves,38,43,11.161385,168
7,Real Madrid,38,86,11.380770,381
8,Getafe,38,51,11.512525,109
9,Atletico Madrid,38,69,11.675792,351


In [9]:
ppda_q = teams['ppda_promedio'].quantile([0.25, 0.5, 0.75])
dc_q = teams['deep_completions_total'].quantile([0.25, 0.5, 0.75])
print('PPDA por cuartiles (más bajo = más presión):')
print(ppda_q.to_string())
print()
print('Deep completions por cuartiles (más alto = más penetración):')
print(dc_q.to_string())

PPDA por cuartiles (más bajo = más presión):
0.25    10.344785
0.50    12.401104
0.75    13.694633

Deep completions por cuartiles (más alto = más penetración):
0.25    181.50
0.50    207.00
0.75    262.25


**Umbrales para los prompts del Scout:**

- `presión alta`: PPDA ≤ Q25
- `presión media`: Q25 < PPDA ≤ Q75
- `presión baja`: PPDA > Q75

- `ataque dominante`: deep_completions ≥ Q75
- `ataque medio`: Q25 ≤ deep_completions < Q75
- `ataque limitado`: deep_completions < Q25

## 5. Tabla agregada — sanity check vs la realidad

In [10]:
teams.sort_values('puntos', ascending=False).reset_index(drop=True)

,team,partidos,puntos,xpuntos,goles_a_favor,goles_en_contra,xg_favor,xg_contra,np_xg_favor,ppda_promedio,deep_completions_total
0,Barcelona,38,94,79.0609,95,36,99.667925,50.710138,92.235163,7.313716,438
1,Real Madrid,38,86,75.9341,77,35,81.485508,44.985830,70.966896,11.380770,381
2,Villarreal,38,72,63.2759,72,46,68.839659,53.666982,62.893443,16.376609,289
3,Atletico Madrid,38,69,61.8281,62,44,68.105439,53.138565,65.132349,11.675792,351
4,Real Betis,38,60,59.9613,59,48,61.297745,50.321923,59.067918,15.743586,262
5,Celta Vigo,38,54,51.3689,53,48,51.991857,56.344472,46.045642,14.510581,263
6,Getafe,38,51,42.9512,32,38,33.962757,48.312659,30.978261,11.512525,109
7,Rayo Vallecano,38,50,55.8956,41,44,59.976055,56.343950,57.002933,8.865585,185
8,Valencia,38,49,53.5428,46,55,53.954595,55.530233,47.265098,13.207327,189
9,Espanyol,38,46,44.1313,43,55,52.380573,62.670768,48.664304,13.126416,171


## Conclusiones para el diseño de los agentes

1. **Datos limpios** — sin NaN ni duplicados; schedule completo.
2. **Posiciones de Understat son agregadas por temporada**, no por partido. Para alineaciones concretas vamos a tirar de `soccerdata.ESPN.read_lineup()` cuando construyamos el Coach.
3. **Filtro de jugadores relevantes**: `minutes >= 500` deja a los regulares. El Coach debe usar este filtro para no proponer suplentes con 30 minutos en toda la temporada.
4. **Umbrales tácticos calculados** (PPDA y deep_completions por cuartiles) listos para meter en los prompts del Scout como rangos calibrados a La Liga 25-26.
5. **Siguiente paso natural**: construir el agente Scout con `get_team_profile(team)` y `get_recent_matches(team, n=5)`.